# Phase 1 — Python Language Fundamentals

**Outcome:** predict the output of any core Python snippet and explain it in terms of
objects, names, types and frames; use every built-in data type and its API fluently.

| Unit | Topic |
| --- | --- |
| 1.1 | How Python executes code |
| 1.2 | Objects, names and mutability |
| 1.3 | Numeric and boolean data types |
| 1.4 | Text and binary data types |
| 1.5 | Collection data types I: lists, tuples and ranges |
| 1.6 | Collection data types II: dictionaries, sets and the `collections` module |
| 1.7 | Copying: shallow copies, deep copies and structured clones |
| 1.8 | Operators, expressions and control flow |
| 1.9 | Iteration, comprehensions and built-in functions |
| 1.10 | Functions: definition and arguments |
| 1.11 | Scope, namespaces and closures |
| 1.12 | Decorators, `functools` and the call stack |
| 1.13 | Iterators, generators and `itertools` |
| 1.14 | Exceptions and context managers |
| Project | Log analyzer |

**How to use this notebook**

- Select the **Python 3.15 (course)** kernel, then run the *Shared setup* cell first.
- Each unit starts with its own setup cell, so you can jump straight to any unit.
- *Predict the output* questions hide the answer in a collapsible block: decide first,
  then run the proof cell.
- Labs: replace the `raise NotImplementedError` in each stub, then run the check cell.
  Until then the checks print FAIL; that is expected.
- Features newer than Python 3.12 are labelled (*New in 3.13*, *Changed in 3.15*, …).

## Shared setup

This cell runs first in every notebook of the course. It imports common modules and
defines three helpers used everywhere:

| Helper | What it does |
| --- | --- |
| `run_checks(test_cases)` | runs `(label, thunk, expected)` cases, prints PASS/FAIL |
| `show_size(label, obj)` | prints `sys.getsizeof(obj)`, type name and `id()` |
| `find_repo_root()` | finds the folder holding `pyproject.toml` |

`run_checks` catches every exception, so a failing check never stops Run All.

> **Note:** each unit re-imports what it needs, so you can run any unit on its own
> after this cell.

In [1]:
import inspect
import os
import platform
import sys
from collections.abc import Callable
from pathlib import Path

type TestCase = tuple[str, Callable[[], object], object]


def run_checks(test_cases: list[TestCase]) -> None:
    """Run (label, thunk, expected) cases and print PASS/FAIL for each. Never raises."""
    passed_count = 0
    for label, thunk, expected in test_cases:
        try:
            actual = thunk()
        except Exception as error:  # a check cell must never stop Run All
            print(f"FAIL  {label}: raised {type(error).__name__}: {error}")
            continue
        if actual == expected:
            passed_count += 1
            print(f"PASS  {label}")
        else:
            print(f"FAIL  {label}: expected {expected!r}, got {actual!r}")
    print(f"{passed_count}/{len(test_cases)} checks passed")


def show_size(label: str, obj: object) -> None:
    """Print the shallow size in bytes, the type and the id of an object."""
    print(f"{label}: {sys.getsizeof(obj)} bytes, type={type(obj).__name__}, id={id(obj)}")


def find_repo_root(start: Path | None = None) -> Path:
    """Walk up from start (default: the working directory) to the folder holding pyproject.toml."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found above the working directory")


print(f"Python {platform.python_version()} ({sys.implementation.name}) on {platform.system()}")
print(f"Repository root: {find_repo_root().name}/")

Python 3.15.1 (cpython) on Linux
Repository root: python-course/


## Unit 1.1 — How Python Executes Code

**Objectives**

- Explain what "compiled" and "interpreted" each mean for CPython.
- Follow one line of source through every stage: tokens, AST, symbol table, code
  object, bytecode, evaluation.
- Find and recognise the cached bytecode files Python writes in `__pycache__`.
- Tell statements from expressions and read Python's block syntax.

**JS bridge**

| JavaScript concept | Python equivalent | The difference that bites |
| --- | --- | --- |
| V8 Ignition bytecode | CPython bytecode (`dis.dis`) | You can print CPython's bytecode from inside the program |
| TurboFan optimising JIT | Specializing adaptive interpreter (3.11+) and experimental JIT (3.13+) | CPython's JIT is off by default; most speed-ups come from specialisation |
| Esprima / acorn AST | `ast` module (built in) | The AST is part of the standard library |
| V8 code cache | `__pycache__/*.pyc` | The cache is a file next to your source, one per interpreter version |
| `{ … }` blocks, semicolons | Indentation, newlines | Indentation is syntax, not style |

In [2]:
import ast
import dis
import importlib
import importlib.util
import io
import os
import subprocess
import symtable
import sys
import tokenize
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Walk up from start (default: the working directory) to the folder holding pyproject.toml."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found above the working directory")


repo_root = find_repo_root()
scratch_dir = repo_root / "notebooks" / "_scratch"
scratch_dir.mkdir(parents=True, exist_ok=True)
print(f"{sys.implementation.name=}, {sys.version_info[:2]=}")

sys.implementation.name='cpython', sys.version_info[:2]=(3, 15)


### 1.1.1 Compiled and interpreted

CPython does both. It **compiles** the whole source file to **bytecode** (instructions
for an imaginary machine), then **interprets** that bytecode in a loop. The imaginary
machine is a **stack-based virtual machine**: instructions push values onto a stack,
and operators pop their operands off it and push the result.

The cell disassembles a one-line function. Read it as a stack machine:

| Instruction | Stack after it |
| --- | --- |
| load `price` | `[price]` |
| load `tax` | `[price, tax]` |
| `BINARY_OP (+)` | `[price + tax]` |
| `RETURN_VALUE` | `[]` (value returned) |

> **Note:** exact instruction names change between Python versions (3.15 fuses the two
> loads into one `LOAD_FAST_BORROW_LOAD_FAST_BORROW` instruction). The shape stays the same.

In [3]:
def add_tax(price: int, tax: int) -> int:
    """Return price plus tax."""
    return price + tax


dis.dis(add_tax)

  1           RESUME                   0

  3           LOAD_FAST_BORROW_LOAD_FAST_BORROW 1 (price, tax)
              BINARY_OP                0 (+)
              RETURN_VALUE


**Specializing adaptive interpreter (new in 3.11, PEP 659).** After a function has run
a few times, CPython rewrites ("quickens") its bytecode in place with versions
specialised for the types it actually saw. `BINARY_OP (+)` becomes
`BINARY_OP_ADD_INT` when both operands were always `int`. If a different type turns up,
the instruction falls back to the generic version.

`dis.dis(..., adaptive=True)` shows the specialised bytecode. The cell calls `add_tax`
1000 times with integers first.

In [4]:
for _ in range(1_000):
    add_tax(100, 18)

dis.dis(add_tax, adaptive=True)

  1           RESUME_CHECK             0

  3           LOAD_FAST_BORROW_LOAD_FAST_BORROW 1 (price, tax)
              BINARY_OP_ADD_INT        0 (+)
              RETURN_VALUE


**JIT compiler (experimental, new in 3.13; faster in 3.14 and 3.15, PEP 744).** On top
of specialisation, CPython can compile hot traces of specialised bytecode to machine
code. The JIT is built in to the uv builds of 3.15 but **off by default**; set the
environment variable `PYTHON_JIT=1` to turn it on. `sys._jit` (3.14+) reports the state.

The cell checks this kernel, then starts a child interpreter with `PYTHON_JIT=1`.
Interpreters built without the JIT print a skip message.

In [5]:
if hasattr(sys, "_jit"):
    print(f"{sys._jit.is_available()=}  (built with the JIT)")
    print(f"{sys._jit.is_enabled()=}  (turned on in this kernel)")
    child_result = subprocess.run(
        [sys.executable, "-c", "import sys; print(sys._jit.is_enabled())"],
        env={**os.environ, "PYTHON_JIT": "1"},
        capture_output=True,
        text=True,
        check=True,
    )
    print(f"with PYTHON_JIT=1: is_enabled() -> {child_result.stdout.strip()}")
else:
    print("Skipped: sys._jit needs Python 3.14+")

sys._jit.is_available()=True  (built with the JIT)
sys._jit.is_enabled()=False  (turned on in this kernel)
with PYTHON_JIT=1: is_enabled() -> True


> **JS bridge:** V8 works the same way in outline. **Ignition** compiles JavaScript to
> bytecode and interprets it; **TurboFan** compiles hot functions to optimised machine
> code using type feedback. CPython's *adaptive interpreter* plays the role of
> Ignition's type feedback, and its JIT is a much younger, simpler TurboFan.

### 1.1.2 The pipeline

```mermaid
flowchart LR
    source["source text"] --> tokenizer["tokenizer"]
    tokenizer -- "tokens" --> parser["PEG parser"]
    parser -- "AST" --> compiler["compiler + symbol table"]
    compiler -- "code object (bytecode)" --> eval_loop["eval loop (VM)"]
    eval_loop --> result["side effects / result"]
```

<details><summary>ASCII fallback</summary>

```text
source --tokenize--> tokens --PEG parser--> AST --compiler (symbol table)--> code object --eval loop--> result
```

</details>

| Stage | What it produces | Module to look at it |
| --- | --- | --- |
| Tokenizer | `NAME`, `OP`, `NUMBER`… tokens | `tokenize` |
| PEG parser (3.9+, PEP 617) | Abstract syntax tree | `ast.parse`, `ast.dump` |
| Symbol table | Which names are local, global, free | `symtable` |
| Compiler | A **code object**: bytecode + constants + names | `compile`, `code.co_*` |
| Eval loop | Runs the bytecode on the VM | `exec`, `eval`, `dis` |

**Stage 1: tokenizer.** It splits source text into tokens.

In [6]:
source_line = "total_in_rupees = price * 2 + 18\n"
for token in tokenize.generate_tokens(io.StringIO(source_line).readline):
    print(f"{tokenize.tok_name[token.type]:<10} {token.string!r}")

NAME       'total_in_rupees'
OP         '='
NAME       'price'
OP         '*'
NUMBER     '2'
OP         '+'
NUMBER     '18'
NEWLINE    '\n'
ENDMARKER  ''


**Stage 2: parser.** It builds an **abstract syntax tree** from the tokens.
`ast.dump(ast.parse(...))` prints the tree. Notice that operator precedence is already
resolved: `price * 2` is nested inside the `+`.

In [7]:
print(ast.dump(ast.parse(source_line), indent=2))

Module(
  body=[
    Assign(
      targets=[
        Name(id='total_in_rupees', ctx=Store())],
      value=BinOp(
        left=BinOp(
          left=Name(id='price'),
          op=Mult(),
          right=Constant(value=2)),
        op=Add(),
        right=Constant(value=18)))])


**Stage 3: symbol table.** Before generating bytecode, the compiler decides the scope of
every name. Inside a function, a name that is assigned is **local**; a name that is only
read is **global** (or a closure variable). This is decided at compile time, which is
why Unit 1.11's `UnboundLocalError` exists.

In [8]:
function_source = """
def add_gst(price):
    taxed_price = price * gst_rate
    return taxed_price
"""
module_table = symtable.symtable(function_source, "<demo>", "exec")
function_table = module_table.lookup("add_gst").get_namespace()
print(f"{function_table.get_parameters()=}")
print(f"{function_table.get_locals()=}")
print(f"{function_table.get_globals()=}")

function_table.get_parameters()=('price',)
function_table.get_locals()=('price', 'taxed_price')
function_table.get_globals()=('gst_rate',)


**Stage 4: compiler.** `compile(source, filename, mode)` returns a **code object**. The
mode decides what source is allowed: `"exec"` for statements (a module), `"eval"` for a
single expression. The code object holds the bytecode (`co_code`) and the tables it
refers to.

In [9]:
module_code = compile(source_line, "<demo>", "exec")
expression_code = compile("price * 2 + 18", "<demo>", "eval")
print(f"{type(module_code).__name__=}")
print(f"{module_code.co_names=}")
print(f"{module_code.co_consts=}")
print(f"{len(module_code.co_code)=} bytes of bytecode")
print(f"{expression_code.co_names=}")

type(module_code).__name__='code'
module_code.co_names=('price', 'total_in_rupees')
module_code.co_consts=(2,)
len(module_code.co_code)=40 bytes of bytecode
expression_code.co_names=('price',)


`dis.dis` turns the code object's bytes back into readable instructions. At module level,
names are stored with `STORE_NAME` (a dict write) instead of `STORE_FAST` (a slot in the
frame), which is one reason code inside functions runs faster.

In [10]:
dis.dis(module_code)

  0           RESUME                   0

  1           LOAD_NAME                0 (price)
              LOAD_SMALL_INT           2
              BINARY_OP                5 (*)
              LOAD_SMALL_INT          18
              BINARY_OP                0 (+)
              STORE_NAME               1 (total_in_rupees)
              LOAD_COMMON_CONSTANT     7 (None)
              RETURN_VALUE


**Stage 5: eval loop.** `exec(code, namespace)` runs a code object with a dict you
supply as its global namespace. The assignment lands in that dict, not in the notebook's
globals. `eval` runs an expression and returns its value.

In [11]:
demo_namespace: dict[str, object] = {"price": 500}
exec(module_code, demo_namespace)
print(f"{demo_namespace['total_in_rupees']=}")
print(f"{'total_in_rupees' in globals()=}")
print(f"{eval(expression_code, {'price': 10})=}")

demo_namespace['total_in_rupees']=1018
'total_in_rupees' in globals()=False
eval(expression_code, {'price': 10})=38


> **Warning:** never `exec` or `eval` text that came from a user. It runs with the
> full power of your program. Use `ast.literal_eval` to parse literals safely.

### 1.1.3 Bytecode caching

Compiling is not free, so when a module is **imported**, CPython saves the code object
to `__pycache__/<module>.cpython-315.pyc` next to the source. The next import loads
the `.pyc` directly, unless the source changed. (The script you run directly, and
notebook cells, are never cached.)

The cell writes a small module into `notebooks/_scratch/` and imports it.
`importlib.util.cache_from_source` computes where its cache file goes.

In [12]:
module_path = scratch_dir / "price_rules.py"
module_path.write_text(
    "GST_RATE = 18\n\n\ndef with_gst(price: int) -> float:\n"
    '    """Return price including GST."""\n'
    "    return price * (100 + GST_RATE) / 100\n",
    encoding="utf-8",
)
if str(scratch_dir) not in sys.path:
    sys.path.insert(0, str(scratch_dir))
importlib.invalidate_caches()
price_rules = importlib.import_module("price_rules")  # same as: import price_rules

cache_path = Path(importlib.util.cache_from_source(str(module_path)))
print(f"{price_rules.with_gst(1000)=}")
print(f"{cache_path.relative_to(scratch_dir)=}")
print(f"{cache_path.exists()=}")

price_rules.with_gst(1000)=1180.0
cache_path.relative_to(scratch_dir)=PosixPath('__pycache__/price_rules.cpython-315.pyc')
cache_path.exists()=True


Every `.pyc` starts with a 4-byte **magic number** that changes whenever the bytecode
format changes. A `.pyc` with the wrong magic number is ignored and rebuilt, which is
why each Python version gets its own file name tag (`sys.implementation.cache_tag`).
The next 12 bytes hold flags and the source's modification time and size.

In [13]:
pyc_header = cache_path.read_bytes()[:16]
print(f"{importlib.util.MAGIC_NUMBER=}")
print(f"{pyc_header[:4]=}")
print(f"{pyc_header[:4] == importlib.util.MAGIC_NUMBER=}")
print(f"{sys.implementation.cache_tag=}")

importlib.util.MAGIC_NUMBER=b'R\x0e\r\n'
pyc_header[:4]=b'R\x0e\r\n'
pyc_header[:4] == importlib.util.MAGIC_NUMBER=True
sys.implementation.cache_tag='cpython-315'


> **Note:** `__pycache__/` belongs in `.gitignore` (this repository has it). Set
> `PYTHONDONTWRITEBYTECODE=1` to stop Python writing caches, e.g. in read-only containers.

### 1.1.4 Statements vs expressions

An **expression** produces a value (`price * 2`, `f(x)`, `a if c else b`). A
**statement** does something (`x = 1`, `if`, `for`, `def`, `import`, `return`). Every
expression can be used as a statement, but not the reverse: `eval` accepts only
expressions. The AST labels each kind.

In [14]:
sample_module = ast.parse("discount = 10\nprice * 2\nif discount:\n    pass")
for node in sample_module.body:
    print(f"{type(node).__name__:<6} {ast.unparse(node)!r}")

try:
    eval("discount = 10")
except SyntaxError as error:
    print(f"{type(error).__name__}: {error.msg}")

Assign 'discount = 10'
Expr   'price * 2'
If     'if discount:\n    pass'
SyntaxError: invalid syntax


**Indentation is block syntax.** A colon opens a block, and the indented lines below it
form the body; there are no braces. PEP 8 uses 4 spaces. Wrong indentation is a
*compile-time* error (`IndentationError`, a subclass of `SyntaxError`), so the cell
compiles bad source as a string to catch it.

In [15]:
badly_indented_source = "if True:\nprint('missing indent')\n"
try:
    compile(badly_indented_source, "<demo>", "exec")
except IndentationError as error:
    print(f"{type(error).__name__}: {error.msg} (line {error.lineno})")
print(f"{issubclass(IndentationError, SyntaxError)=}")

IndentationError: expected an indented block after 'if' statement on line 1 (line 2)
issubclass(IndentationError, SyntaxError)=True


**`pass`** is the statement that does nothing. Python needs it where a block is required
but you have nothing to put there yet (JS would use `{}`).

In [16]:
def not_written_yet() -> None:
    """Placeholder whose body is intentionally empty."""
    pass


class EmptyMarker:
    pass


print(f"{not_written_yet()=}, {EmptyMarker.__name__=}")

not_written_yet()=None, EmptyMarker.__name__='EmptyMarker'


**Line continuation.** A newline ends a statement *unless* it is inside `()`, `[]` or
`{}`. Wrap long expressions in brackets instead of ending a line with a backslash:

```python
order_total = 499 + \
    150          # explicit continuation: breaks if a space follows the backslash
```

`ruff format` (and Black) never produce backslash continuations; they add parentheses.

In [17]:
line_item_prices = [
    499,
    150,
    1_200,
]
order_total = (
    sum(line_item_prices)
    + 40  # delivery charge
    - 100  # coupon
)
print(f"{order_total=}, {line_item_prices=}")

order_total=1789, line_item_prices=[499, 150, 1200]


#### Under the hood: one code object per function

A function is a thin wrapper around a code object compiled once, when its `def` ran.
`__code__` exposes it, with the variable names and constants the bytecode uses.

In [18]:
add_tax_code = add_tax.__code__
print(f"{add_tax_code.co_name=}")
print(f"{add_tax_code.co_varnames=}")
print(f"{add_tax_code.co_argcount=}")
print(f"{add_tax_code.co_consts=}")
print(f"{add_tax.__code__ is add_tax_code=} (compiled once, reused on every call)")

add_tax_code.co_name='add_tax'
add_tax_code.co_varnames=('price', 'tax')
add_tax_code.co_argcount=2
add_tax_code.co_consts=('Return price plus tax.',)
add_tax.__code__ is add_tax_code=True (compiled once, reused on every call)


#### Predict the output

**Predict the output.**

```python
namespace = {}
exec("x = 2\ny = x * 21", namespace)
print(namespace["y"], "y" in globals())
```

<details><summary>Answer</summary>

`42 False`. `exec` used `namespace` as the global namespace of the code it ran, so both
`x` and `y` were stored there and not in the notebook's globals.

</details>

Run the next cell to check your prediction.

In [19]:
namespace: dict[str, object] = {}
exec("x = 2\ny = x * 21", namespace)
print(namespace["y"], "y" in globals())

42 False


**Predict the output.**

```python
print(type(ast.parse("1 + 2").body[0]).__name__)
```

<details><summary>Answer</summary>

`Expr`. `ast.parse` defaults to `"exec"` mode, so the source is a module whose body is a
list of **statements**. A bare expression becomes an *expression statement* (`Expr`)
that wraps the `BinOp`.

</details>

Run the next cell to check your prediction.

In [20]:
print(type(ast.parse("1 + 2").body[0]).__name__)
print(type(ast.parse("1 + 2").body[0].value).__name__)

Expr
BinOp


### Recap — Unit 1.1

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| Compiled *and* interpreted | Source → bytecode once, then a stack VM runs it | V8 Ignition |
| Adaptive interpreter | Hot bytecode is specialised for the types it sees (3.11+) | Inline caches / type feedback |
| JIT | Experimental, off by default, enable with `PYTHON_JIT=1` (3.13+) | TurboFan |
| Pipeline | tokenize → PEG parser → AST → symbol table → code object → eval loop | parse → AST → bytecode → execute |
| `exec` / `eval` | Run code in a namespace dict you control; never on untrusted input | `eval`, `new Function` |
| `__pycache__` | Imported modules cache bytecode in `.pyc`, checked by magic number | V8 code cache |
| Statement vs expression | Only expressions have values; `eval` accepts only expressions | Same split |
| Blocks | Indentation + colon; `pass` for an empty body; wrap long lines in brackets | `{ }` and `;` |

**Further reading**

- [`dis` — Disassembler](https://docs.python.org/3/library/dis.html),
  [`ast`](https://docs.python.org/3/library/ast.html),
  [`symtable`](https://docs.python.org/3/library/symtable.html),
  [`tokenize`](https://docs.python.org/3/library/tokenize.html)
- [Built-in `compile`, `exec`, `eval`](https://docs.python.org/3/library/functions.html#compile),
  [`importlib.util.cache_from_source`](https://docs.python.org/3/library/importlib.html#importlib.util.cache_from_source)
- [Language reference: lexical analysis (indentation, line joining)](https://docs.python.org/3/reference/lexical_analysis.html)
- [PEP 617 — PEG parser](https://peps.python.org/pep-0617/),
  [PEP 659 — Specializing adaptive interpreter](https://peps.python.org/pep-0659/),
  [PEP 744 — JIT compilation](https://peps.python.org/pep-0744/),
  [PEP 3147 — `__pycache__` directories](https://peps.python.org/pep-3147/)

## Unit 1.2 — Objects, Names and Mutability

**Objectives**

- Describe every value as an object with an identity, a type and a value.
- Explain variables as names bound to objects in a namespace dict.
- Predict when a function call changes the caller's data.
- Classify built-in types as mutable or immutable, and use `is` vs `==` correctly.
- Explain why names have no type but objects do.

**JS bridge**

| JavaScript concept | Python equivalent | The difference that bites |
| --- | --- | --- |
| Primitives (`number`, `string`) vs objects | Everything is an object | `int` and `str` are objects too, just immutable ones |
| `let x = obj` copies a reference | `x = obj` binds a name | Same idea, but it applies to *every* value |
| `===` | `==` (value) and `is` (identity) | `is` on numbers or strings depends on caching; never use it for that |
| `Object.is(a, b)` | `a is b` | `is` compares identity, not value |
| `typeof x` | `type(x)` / `isinstance(x, T)` | `isinstance` respects subclasses |
| `delete obj.key` | `del name` | `del` unbinds a name; the object lives on while referenced |

The next cell is this unit's setup.

In [21]:
import inspect
import math
import sys
import types
import warnings


def show_size(label: str, obj: object) -> None:
    """Print the shallow size in bytes, the type and the id of an object."""
    print(f"{label}: {sys.getsizeof(obj)} bytes, type={type(obj).__name__}, id={id(obj)}")


print(f"{sys.version_info[:2]=}")

sys.version_info[:2]=(3, 15)


### 1.2.1 Everything is an object

Every object has three things:

| Property | How to read it | Can it change? |
| --- | --- | --- |
| **Identity** | `id(obj)` (CPython: the memory address) | Never, for the object's lifetime |
| **Type** | `type(obj)` | Never (in practice) |
| **Value** | the object itself, `repr(obj)` | Only for mutable types |

> **Note:** `id()` values differ on every run. The demos compare ids; they never rely on
> the number itself.

In [22]:
price_in_rupees = 499
cart_items = ["pen", "notebook"]
for value in (price_in_rupees, "Asha", cart_items, 3.5, None):
    print(f"type={type(value).__name__:<8} value={value!r:<20} has an id: {id(value) > 0}")

type=int      value=499                  has an id: True
type=str      value='Asha'               has an id: True
type=list     value=['pen', 'notebook']  has an id: True
type=float    value=3.5                  has an id: True
type=NoneType value=None                 has an id: True


Types, functions, modules and classes are objects too: you can put them in a list, pass
them to functions and give them attributes. Even `int` is an object, whose type is
`type`.

In [23]:
def apply_discount(amount: float) -> float:
    """Return the amount with a 10% discount."""
    return amount * 0.9


first_class_values = [int, apply_discount, math, type, len]
for value in first_class_values:
    print(f"{getattr(value, '__name__', value)!s:<16} is a {type(value).__name__}")

print(f"{type(int) is type=}, {type(type) is type=}")
print(f"{inspect.isclass(int)=}, {inspect.isfunction(apply_discount)=}, {inspect.ismodule(math)=}")
apply_discount.author = "Asha"  # functions accept new attributes like any object
print(f"{apply_discount.author=}")

int              is a type
apply_discount   is a function
math             is a module
type             is a type
len              is a builtin_function_or_method
type(int) is type=True, type(type) is type=True
inspect.isclass(int)=True, inspect.isfunction(apply_discount)=True, inspect.ismodule(math)=True
apply_discount.author='Asha'


### 1.2.2 Names and binding

A variable is a **name**: a key in a namespace dict, bound to an object. At module level
(and in a notebook) that dict is `globals()`. Writing to the dict is the same as
assigning a variable.

In [24]:
delivery_fee = 40
module_namespace = globals()
print(f"{type(module_namespace).__name__=}")
print(f"{module_namespace['delivery_fee']=}")

module_namespace["service_fee"] = 15  # same effect as: service_fee = 15
print(f"{service_fee=}")  # noqa: F821  (created through the dict, invisible to linters)

type(module_namespace).__name__='dict'
module_namespace['delivery_fee']=40
service_fee=15


**Assignment never copies.** `b = a` binds a second name to the *same* object. If the
object is mutable, a change through one name is visible through the other.

```mermaid
flowchart LR
    subgraph names ["globals()"]
        morning["morning_orders"]
        alias["evening_orders"]
    end
    list_object["list object<br/>['idli', 'dosa', 'vada']"]
    morning --> list_object
    alias --> list_object
```

<details><summary>ASCII fallback (Python Tutor style)</summary>

```text
Frames            Objects
+----------------+
| morning_orders | ----+
| evening_orders | ----+--> list ['idli', 'dosa', 'vada']
+----------------+
```

</details>

In [25]:
morning_orders = ["idli", "dosa"]
evening_orders = morning_orders  # a second name, not a second list
evening_orders.append("vada")
print(f"{morning_orders=}")
print(f"{evening_orders is morning_orders=}")

morning_orders=['idli', 'dosa', 'vada']
evening_orders is morning_orders=True


**`del` removes a name, not the object.** The object survives as long as something
still refers to it. `sys.getrefcount` counts the references (it reports one extra for
its own argument). CPython frees the object when the count reaches zero.

In [26]:
reference_count_before = sys.getrefcount(morning_orders)
del morning_orders
print(f"{'morning_orders' in globals()=}")
print(f"{evening_orders=}  (the list is still alive)")
print(f"{sys.getrefcount(evening_orders) == reference_count_before - 1=}")
try:
    print(morning_orders)  # noqa: F821
except NameError as error:
    print(f"{type(error).__name__}: {error}")

'morning_orders' in globals()=False
evening_orders=['idli', 'dosa', 'vada']  (the list is still alive)
sys.getrefcount(evening_orders) == reference_count_before - 1=True
NameError: name 'morning_orders' is not defined


### 1.2.3 Call by object reference

Calling a function binds each parameter name to the **same object** the caller passed
(sometimes called *call by sharing*). Two consequences:

| Inside the function | Effect on the caller |
| --- | --- |
| **Mutate** the object (`items.append(...)`) | Visible: it is the same object |
| **Rebind** the parameter (`items = [...]`) | Invisible: only the local name moved |

In [27]:
def add_free_gift(order_items: list[str]) -> None:
    """Mutate the caller's list by appending a gift."""
    order_items.append("free sticker")


def replace_order(order_items: list[str]) -> None:
    """Rebind the local name only; the caller's list is untouched."""
    order_items = ["something else"]
    print(f"  inside replace_order: {order_items=}")


customer_order = ["tea"]
add_free_gift(customer_order)
print(f"after add_free_gift: {customer_order=}")
replace_order(customer_order)
print(f"after replace_order: {customer_order=}")

after add_free_gift: customer_order=['tea', 'free sticker']
  inside replace_order: order_items=['something else']
after replace_order: customer_order=['tea', 'free sticker']


The same rule applies to immutable arguments: `count += 1` inside a function rebinds the
local name to a new `int`, so the caller sees no change. Return the new value instead.

> **JS bridge:** Python has **no primitives**. In JS, numbers and strings are passed by
> value and objects by reference-copy. In Python *everything* is passed the same way;
> `int` and `str` only *look* like they are passed by value because they cannot be
> mutated.

In [28]:
def increment(counter: int) -> int:
    """Return counter + 1; rebinding the parameter does not touch the caller."""
    counter += 1
    return counter


visit_count = 10
new_visit_count = increment(visit_count)
print(f"{visit_count=}, {new_visit_count=}")

visit_count=10, new_visit_count=11


### 1.2.4 Mutable vs immutable types

| Immutable (value fixed at creation) | Mutable (value can change in place) |
| --- | --- |
| `int`, `float`, `complex`, `bool` | `list` |
| `decimal.Decimal`, `fractions.Fraction` | `dict` |
| `str` | `set` |
| `bytes` | `bytearray` |
| `tuple` (its items may be mutable) | `memoryview` over a writable buffer |
| `frozenset` | `collections.deque`, `array.array` |
| `frozendict` (new in 3.15) | instances of most user-defined classes |
| `range`, `NoneType`, functions' code objects | `types.SimpleNamespace` |

"Changing" an immutable object really creates a **new** object and rebinds the name.
Watch the identity across `+=` (comparing ids, never printing them):

In [29]:
stock_count = 1_000
id_before = id(stock_count)
stock_count += 1
print(f"int:  same object after += ? {id(stock_count) == id_before}")

greeting_text = "Namaste"
id_before = id(greeting_text)
greeting_text += ", Asha"
print(f"str:  same object after += ? {id(greeting_text) == id_before}")

basket = ["rice"]
id_before = id(basket)
basket += ["dal"]  # list += extends in place
print(f"list: same object after += ? {id(basket) == id_before}")

int:  same object after += ? False
str:  same object after += ? False
list: same object after += ? True


The rule of thumb: an immutable type has no method that modifies it. `str.upper()`
returns a new string; `list.sort()` changes the list and returns `None`. Trying to
assign into an immutable object raises `TypeError`.

In [30]:
coordinates = (12.97, 77.59)
try:
    coordinates[0] = 13.0
except TypeError as error:
    print(f"{type(error).__name__}: {error}")

city = "bengaluru"
print(f"{city.upper()=}, {city=}  (unchanged)")
print(f"{basket.sort()=}  (sorted in place, returns None)")

TypeError: 'tuple' object does not support item assignment
city.upper()='BENGALURU', city='bengaluru'  (unchanged)
basket.sort()=None  (sorted in place, returns None)


### 1.2.5 Identity vs equality

| Operator | Question | Implemented by |
| --- | --- | --- |
| `a == b` | Do they have equal **values**? | `a.__eq__(b)`, which each type defines |
| `a is b` | Are they the **same object**? | Identity comparison; cannot be overridden |

Use `is` only for singletons: `None`, `True`, `False`, `...`, and sentinel objects you
create yourself. Use `==` for everything else.

In [31]:
class Price:
    """An amount in paise that compares by value."""

    def __init__(self, paise: int) -> None:
        self.paise = paise

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Price):
            return NotImplemented
        return self.paise == other.paise


first_price = Price(49_900)
second_price = Price(49_900)
print(f"{first_price == second_price=}  (__eq__ compares paise)")
print(f"{first_price is second_price=}  (two distinct objects)")
first_list, second_list = [1, 2], [1, 2]
print(f"{first_list == second_list=}, {first_list is second_list=}")

first_price == second_price=True  (__eq__ compares paise)
first_price is second_price=False  (two distinct objects)
first_list == second_list=True, first_list is second_list=False


**Small-integer cache.** CPython pre-creates small `int` objects and reuses them, so two
names bound to the same small value can be the *same object*. The range changed:

| Python | Cached range |
| --- | --- |
| up to 3.14 | `-5` … `256` |
| **3.15** (*changed in 3.15*) | `-5` … `1024` |

The cell builds numbers at run time with `int("…")`. Literals in one cell would be
merged into a single constant by the compiler and hide the effect. This is an
implementation detail, which is exactly why you must not use `is` for numbers.

In [32]:
for number_text in ["256", "257", "1024", "1025"]:
    first_number, second_number = int(number_text), int(number_text)
    is_same_object = first_number is second_number
    print(f"{number_text:>5}: equal={first_number == second_number}, same object={is_same_object}")
if sys.version_info < (3, 15):
    print("Note: on this Python the cache ends at 256, so 1024 is not shared")

  256: equal=True, same object=True
  257: equal=True, same object=True
 1024: equal=True, same object=True
 1025: equal=True, same object=False


**String interning.** CPython also shares some strings: identifier-like literals and
names are *interned* automatically. Strings built at run time usually are not.
`sys.intern` interns one on request, which can save memory and makes later comparisons
identity checks (used inside dicts of attribute names).

In [33]:
built_status = "".join(["sh", "ipped"])
literal_status = "shipped"
print(f"{built_status == literal_status=}, {built_status is literal_status=}")

interned_status = sys.intern(built_status)
print(f"{interned_status is literal_status=}  (sys.intern returned the shared copy)")

built_status == literal_status=True, built_status is literal_status=False
interned_status is literal_status=True  (sys.intern returned the shared copy)


Because results like these vary by version and context, the compiler warns when you use
`is` with a literal. The cell compiles such code and captures the **`SyntaxWarning`**
(new in 3.8) instead of letting it print.

In [34]:
suspicious_source = "order_count = 5\nprint(order_count is 5)"
with warnings.catch_warnings(record=True) as caught_warnings:
    warnings.simplefilter("always")
    compile(suspicious_source, "<demo>", "exec")
for caught_warning in caught_warnings:
    print(f"{caught_warning.category.__name__}: {caught_warning.message}")

### 1.2.6 Dynamic typing

**Names have no types; objects do.** One name can be rebound to objects of different
types. Type hints (`price: int`) document intent and help tools such as pyright, but
the interpreter does not enforce them (Phase 4).

In [35]:
shipment_reference = 1042
print(f"{type(shipment_reference).__name__=}")
shipment_reference = "SHIP-1042"
print(f"{type(shipment_reference).__name__=}")

hinted_quantity: int = "three"  # legal at run time: hints are not checked
print(f"{hinted_quantity=}, {type(hinted_quantity).__name__=}")

type(shipment_reference).__name__='int'
type(shipment_reference).__name__='str'
hinted_quantity='three', type(hinted_quantity).__name__='str'


**`type()` vs `isinstance()`.** `type(x) is T` matches one exact class.
`isinstance(x, T)` also accepts subclasses, and takes a tuple (or a `X | Y` union) of
types. Prefer `isinstance` for checks.

In [36]:
class DiscountedPrice(Price):
    """A Price subclass."""


sale_price = DiscountedPrice(39_900)
print(f"{type(sale_price) is Price=}")
print(f"{isinstance(sale_price, Price)=}")
print(f"{isinstance(3.5, int | float)=}")
print(f"{isinstance(True, int)=}  (bool is a subclass of int; see 1.3.6)")

type(sale_price) is Price=False
isinstance(sale_price, Price)=True
isinstance(3.5, int | float)=True
isinstance(True, int)=True  (bool is a subclass of int; see 1.3.6)


**Duck typing preview.** Python code usually does not check types at all: it calls the
method it needs and lets anything that has it work ("if it walks like a duck…").
Phase 2 formalises this with protocols.

In [37]:
class PaperReceipt:
    """A receipt that is printed."""

    def render(self) -> str:
        """Return a description of the receipt."""
        return "Printed receipt"


class EmailReceipt:
    """A receipt that is emailed; unrelated to PaperReceipt."""

    def render(self) -> str:
        """Return a description of the receipt."""
        return "Emailed receipt"


def send_receipt(receipt: PaperReceipt | EmailReceipt) -> None:
    """Works with any object that has a render() method."""
    print(receipt.render())


ad_hoc_receipt = types.SimpleNamespace(render=lambda: "Ad-hoc receipt")
for receipt in (PaperReceipt(), EmailReceipt(), ad_hoc_receipt):
    send_receipt(receipt)

Printed receipt
Emailed receipt
Ad-hoc receipt


#### Under the hood: names live in `__dict__`, objects carry headers

A module's globals are literally its `__dict__`, and an instance's attributes are in its
own `__dict__`. Every object also carries a header (reference count and type pointer),
which is why even `0` costs bytes. `show_size` (from the shared setup) prints the
shallow size.

In [38]:
print(f"{globals() is sys.modules['__main__'].__dict__=}")
print(f"{vars(first_price)=}")
show_size("small int 0", 0)
show_size("None", None)
show_size("empty list", [])

globals() is sys.modules['__main__'].__dict__=True
vars(first_price)={'paise': 49900}
small int 0: 28 bytes, type=int, id=22124928
None: 16 bytes, type=NoneType, id=22109848
empty list: 56 bytes, type=list, id=139868725145984


#### Predict the output

**Predict the output.**

```python
shopping_list = ["milk"]
other_list = shopping_list
other_list = other_list + ["bread"]
print(shopping_list)
```

<details><summary>Answer</summary>

`['milk']`. `other_list + [...]` builds a **new** list and rebinds `other_list` to it;
`shopping_list` still points at the original. Compare `basket += [...]` in 1.2.4,
which extends the list in place.

</details>

Run the next cell to check your prediction.

In [39]:
shopping_list = ["milk"]
other_list = shopping_list
other_list = other_list + ["bread"]
print(shopping_list)

['milk']


**Predict the output.**

```python
def reset(scores):
    scores.clear()
    scores = [100]

team_scores = [7, 9]
reset(team_scores)
print(team_scores)
```

<details><summary>Answer</summary>

`[]`. `clear()` mutates the caller's list; the rebinding on the next line only moves
the local name.

</details>

Run the next cell to check your prediction.

In [40]:
def reset(scores: list[int]) -> None:
    """Clear the caller's list, then rebind the local name."""
    scores.clear()
    scores = [100]


team_scores = [7, 9]
reset(team_scores)
print(team_scores)

[]


### Recap — Unit 1.2

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| Everything is an object | Identity, type and value; classes, functions, modules too | Objects, but JS also has primitives |
| Names and binding | A variable is a key in a namespace dict bound to an object | Variable holds a reference |
| Assignment | Never copies; two names can share one object | Same for objects |
| `del` | Removes a name; the object dies when unreferenced | `delete` removes a property |
| Call by object reference | Mutation is visible to the caller, rebinding is not | Same for objects; primitives copy |
| Mutable vs immutable | `list`/`dict`/`set` change in place; `int`/`str`/`tuple` never do | Objects vs primitives |
| `is` vs `==` | `is` for singletons only; small-int cache is `-5..1024` in 3.15 | `Object.is` vs `===` |
| Dynamic typing | Names untyped, objects typed; prefer `isinstance` and duck typing | `typeof`, `instanceof` |

**Further reading**

- [Data model: objects, values and types](https://docs.python.org/3/reference/datamodel.html#objects-values-and-types)
- [Naming and binding](https://docs.python.org/3/reference/executionmodel.html#naming-and-binding)
- [Programming FAQ: how do I write a function with output parameters?](https://docs.python.org/3/faq/programming.html#how-do-i-write-a-function-with-output-parameters-call-by-reference)
- [`sys.intern`](https://docs.python.org/3/library/sys.html#sys.intern),
  [`isinstance`](https://docs.python.org/3/library/functions.html#isinstance),
  [`id`](https://docs.python.org/3/library/functions.html#id)
- [PEP 8 — comparisons to singletons](https://peps.python.org/pep-0008/#programming-recommendations)

## Unit 1.3 — Numeric and Boolean Data Types

**Objectives**

- Use `int`, `float`, `Decimal`, `Fraction` and `complex`, and choose the right one for
  money, measurements and exact ratios.
- Predict `//`, `%` and `round()` results, including negative numbers and halves.
- Explain why `bool` is an `int`, what `None` is, and which values are falsy.
- Convert between types explicitly, knowing what raises.

**JS bridge**

| JavaScript concept | Python equivalent | The difference that bites |
| --- | --- | --- |
| `number` (one type) | `int` and `float` (two types) | `7 / 2 == 3.5`, but `7 // 2 == 3`; ints never lose precision |
| `BigInt` (`10n`) | `int` | Every `int` is a big int; no suffix, no mixing errors |
| `-7 % 2 === -1` | `-7 % 2 == 1` | `%` takes the sign of the divisor |
| `Math.round(2.5) === 3` | `round(2.5) == 2` | Banker's rounding (round half to even) |
| `1 / 0 === Infinity` | `1 / 0` raises `ZeroDivisionError` | Division by zero is an error |
| `null` and `undefined` | `None` only | Missing dict keys raise `KeyError` |
| `[]` and `{}` are truthy | `[]` and `{}` are falsy | Empty containers are false |
| `1 + "1" === "11"` | `TypeError` | No implicit coercion |

The next cell is this unit's setup.

In [41]:
import cmath
import math
import sys
import timeit
from collections.abc import Callable
from decimal import ROUND_HALF_EVEN, ROUND_HALF_UP, Decimal, getcontext, localcontext
from fractions import Fraction


def show_size(label: str, obj: object) -> None:
    """Print the shallow size in bytes, the type and the id of an object."""
    print(f"{label}: {sys.getsizeof(obj)} bytes, type={type(obj).__name__}, id={id(obj)}")


PAISE = Decimal("0.01")  # quantize() target for rupee amounts
print(f"{sys.version_info[:2]=}, {getcontext().prec=}")

sys.version_info[:2]=(3, 15), getcontext().prec=28


### 1.3.0 Python's built-in data types at a glance

Units 1.3–1.7 teach these types:

| Category | Type | Mutable | Ordered | Hashable | Literal / constructor | Section |
| --- | --- | --- | --- | --- | --- | --- |
| Numeric | `int` | No | — | Yes | `42`, `0xff`, `1_000` | 1.3.1 |
| Numeric | `float` | No | — | Yes | `3.14`, `1e-3` | 1.3.2 |
| Numeric | `complex` | No | — | Yes | `3 + 4j` | 1.3.5 |
| Numeric (stdlib) | `Decimal`, `Fraction` | No | — | Yes | `Decimal("0.1")`, `Fraction(1, 3)` | 1.3.3–1.3.4 |
| Boolean | `bool` | No | — | Yes | `True`, `False` | 1.3.6 |
| Null | `NoneType` | No | — | Yes | `None` | 1.3.7 |
| Text | `str` | No | Yes | Yes | `"hi"`, `f"{name}"` | 1.4 |
| Binary | `bytes` | No | Yes | Yes | `b"hi"` | 1.4.5 |
| Binary | `bytearray` | Yes | Yes | No | `bytearray(b"hi")` | 1.4.5 |
| Binary | `memoryview` | Depends on buffer | Yes | Read-only views only | `memoryview(data)` | 1.4.6 |
| Sequence | `list` | Yes | Yes | No | `[1, 2]` | 1.5 |
| Sequence | `tuple` | No | Yes | If all items are | `(1, 2)`, `(1,)` | 1.5.5 |
| Sequence | `range` | No | Yes | Yes | `range(10)` | 1.5.7 |
| Mapping | `dict` | Yes | Insertion order | No | `{"a": 1}` | 1.6 |
| Mapping | `frozendict` (3.15) | No | Insertion order | If all values are | `frozendict(a=1)` | 1.6.5 |
| Set | `set` | Yes | No | No | `{1, 2}`, `set()` | 1.6.6 |
| Set | `frozenset` | No | No | Yes | `frozenset({1, 2})` | 1.6.6 |

**Hashable** means `hash(obj)` works, so the object can be a dict key or a set member
(1.6.2). The cell checks the *Hashable* column for one sample of each row, including the
"it depends" cases. Unhashable objects raise `TypeError`, except a writable
`memoryview`, which raises `ValueError`. `frozendict` is *new in 3.15*, so it is added
only on 3.15+.

In [42]:
hashability_samples: list[tuple[str, object]] = [
    ("int", 42),
    ("float", 3.14),
    ("complex", 3 + 4j),
    ("Decimal", Decimal("0.1")),
    ("Fraction", Fraction(1, 3)),
    ("bool", True),
    ("NoneType", None),
    ("str", "hi"),
    ("bytes", b"hi"),
    ("bytearray", bytearray(b"hi")),
    ("memoryview (read-only)", memoryview(b"hi")),
    ("memoryview (writable)", memoryview(bytearray(b"hi"))),
    ("list", [1, 2]),
    ("tuple of ints", (1, 2)),
    ("tuple holding a list", (1, [2])),
    ("range", range(10)),
    ("dict", {"a": 1}),
    ("set", {1, 2}),
    ("frozenset", frozenset({1, 2})),
]
if sys.version_info >= (3, 15):
    hashability_samples += [
        ("frozendict of ints", frozendict(a=1)),
        ("frozendict holding a list", frozendict(a=[1])),
    ]
else:
    print("Skipped frozendict rows: new in 3.15")

for label, sample in hashability_samples:
    try:
        hash(sample)
        verdict = "hashable"
    except (TypeError, ValueError) as error:  # writable memoryview raises ValueError
        verdict = f"unhashable ({type(error).__name__})"
    print(f"{label:<26} {verdict}")

int                        hashable
float                      hashable
complex                    hashable
Decimal                    hashable
Fraction                   hashable
bool                       hashable
NoneType                   hashable
str                        hashable
bytes                      hashable
bytearray                  unhashable (TypeError)
memoryview (read-only)     hashable
memoryview (writable)      unhashable (ValueError)
list                       unhashable (TypeError)
tuple of ints              hashable
tuple holding a list       unhashable (TypeError)
range                      hashable
dict                       unhashable (TypeError)
set                        unhashable (TypeError)
frozenset                  hashable
frozendict of ints         hashable
frozendict holding a list  unhashable (TypeError)


### 1.3.1 Integers (`int`)

Python's `int` has **arbitrary precision**: it grows to hold any value, so it never
overflows. There is no separate BigInt type and no `n` suffix. JavaScript's `number`
silently loses integer precision above `2**53`; Python does not.

In [43]:
js_max_safe_integer = 2**53 - 1
print(f"{js_max_safe_integer + 2=}  (exact; JS number would print ...992)")
print(f"{2**100=}")
print(f"{math.factorial(30)=}")
print(f"{float(2**53 + 1) == 2**53=}  (float, like JS number, cannot hold it)")

js_max_safe_integer + 2=9007199254740993  (exact; JS number would print ...992)
2**100=1267650600228229401496703205376
math.factorial(30)=265252859812191058636308480000000
float(2**53 + 1) == 2**53=True  (float, like JS number, cannot hold it)


Internally CPython stores an `int` as an array of **30-bit digits** (`sys.int_info`), so
its memory grows by 4 bytes for every 30 bits of magnitude. The growth table:

In [44]:
print(f"{sys.int_info=}")
print(f"{'value':>14} {'bits':>5} {'bytes':>6}")
for label, number in [
    ("0", 0),
    ("1", 1),
    ("2**30 - 1", 2**30 - 1),
    ("2**30", 2**30),
    ("2**60", 2**60),
    ("2**90", 2**90),
    ("10**100", 10**100),
    ("10**1000", 10**1000),
]:
    print(f"{label:>14} {number.bit_length():>5} {sys.getsizeof(number):>6}")

sys.int_info=sys.int_info(bits_per_digit=30, sizeof_digit=4, default_max_str_digits=4300, str_digits_check_threshold=640)
         value  bits  bytes
             0     0     28
             1     1     28
     2**30 - 1    30     28
         2**30    31     32
         2**60    61     36
         2**90    91     40
       10**100   333     72
      10**1000  3322    468


**Literals.** Decimal, binary (`0b`), octal (`0o`, never a bare leading zero) and
hexadecimal (`0x`). Underscores group digits for readability and are ignored.

In [45]:
population_of_india = 1_428_627_663
file_mode = 0o755
flag_byte = 0b1010_0001
brand_colour = 0xFF9933
print(f"{population_of_india=}, {file_mode=}, {flag_byte=}, {brand_colour=}")
try:
    eval("0755")
except SyntaxError as error:
    print(f"{type(error).__name__}: {error.msg}")

population_of_india=1428627663, file_mode=493, flag_byte=161, brand_colour=16750899
SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers


**Operators.**

| Operator | Meaning | `17 op 5` |
| --- | --- | --- |
| `+ - *` | add, subtract, multiply | `22`, `12`, `85` |
| `/` | true division, **always returns `float`** | `3.4` |
| `//` | floor division: rounds **down** (towards −∞) | `3` |
| `%` | modulo: the result takes the **divisor's** sign | `2` |
| `**` | power | `1419857` |

Floor means *down*, not *towards zero*, so `-7 // 2 == -4`. And `a == (a // b) * b + a % b`
always holds, which forces `-7 % 2 == 1`. JavaScript's `%` is a *remainder* that keeps the
dividend's sign (`-7 % 2 === -1`).

In [46]:
print(f"{17 + 5=}, {17 - 5=}, {17 * 5=}, {17 ** 5=}")
print(f"{17 / 5=}, {10 / 2=}  (always float)")
print(f"{17 // 5=}, {17 % 5=}")
print(f"{-7 // 2=}, {-7 % 2=}  (JS: Math.trunc gives -3, % gives -1)")
print(f"{7 // -2=}, {7 % -2=}")
dividend, divisor = -7, 2
print(f"{(dividend // divisor) * divisor + dividend % divisor == dividend=}")
print(f"{math.fmod(-7, 2)=}  (JS-style remainder, when you need it)")

17 + 5=22, 17 - 5=12, 17 * 5=85, 17 ** 5=1419857
17 / 5=3.4, 10 / 2=5.0  (always float)
17 // 5=3, 17 % 5=2
-7 // 2=-4, -7 % 2=1  (JS: Math.trunc gives -3, % gives -1)
7 // -2=-4, 7 % -2=-1
(dividend // divisor) * divisor + dividend % divisor == dividend=True
math.fmod(-7, 2)=-1.0  (JS-style remainder, when you need it)


**Functions and methods.** Each gets its own demo below.

| Name | What it does |
| --- | --- |
| `abs(x)` | absolute value |
| `divmod(a, b)` | `(a // b, a % b)` in one call |
| `pow(base, exp, mod)` | modular power, fast even for huge exponents |
| `round(x, ndigits)` | rounds to a multiple of `10**-ndigits` (round half to even) |
| `int.bit_length()` | bits needed, ignoring the sign |
| `int.bit_count()` | number of 1 bits (*popcount*) |
| `int.to_bytes` / `int.from_bytes` | convert to and from raw bytes |
| `bin` / `oct` / `hex` | text in base 2 / 8 / 16 |
| `int(text, base)` | parse text in any base 2–36 |
| `int.is_integer()` | always `True`; exists so code can treat `int` and `float` alike |

`abs` works on every numeric type.

In [47]:
account_balance_change = -2_500
print(f"{abs(account_balance_change)=}, {abs(-3.5)=}, {abs(Decimal('-0.10'))=}")

abs(account_balance_change)=2500, abs(-3.5)=3.5, abs(Decimal('-0.10'))=Decimal('0.10')


`divmod` returns quotient and remainder together, e.g. to split minutes into hours.

In [48]:
hours, minutes = divmod(135, 60)
print(f"{hours=}, {minutes=}")
print(f"{divmod(-7, 2)=}")

hours=2, minutes=15
divmod(-7, 2)=(-4, 1)


Three-argument `pow(base, exp, mod)` computes `base ** exp % mod` without building the
huge intermediate number. It is the core of RSA. With `exp=-1` it returns the modular
inverse (3.8+).

In [49]:
print(f"{pow(2, 10)=}")
print(f"{pow(7, 10**18, 13)=}  (instant; 7 ** 10**18 would never finish)")
print(f"{pow(3, -1, 11)=}  (3 * 4 % 11 == 1)")

pow(2, 10)=1024
pow(7, 10**18, 13)=9  (instant; 7 ** 10**18 would never finish)
pow(3, -1, 11)=4  (3 * 4 % 11 == 1)


`round(int, ndigits)` with a **negative** `ndigits` rounds to tens, hundreds, …. Ties
go to the even neighbour, so `1250` becomes `1200`.

In [50]:
print(f"{round(1_234, -2)=}, {round(1_250, -2)=}, {round(1_350, -2)=}")
print(f"{round(7)=}, {type(round(7.6)).__name__=}  (round() with no ndigits returns int)")

round(1_234, -2)=1200, round(1_250, -2)=1200, round(1_350, -2)=1400
round(7)=7, type(round(7.6)).__name__='int'  (round() with no ndigits returns int)


`bit_length()` is the number of bits needed for the magnitude; `bit_count()` counts the
ones (*new in 3.10*).

In [51]:
sample_number = 0b1011_0000
print(f"{sample_number=}, {sample_number.bit_length()=}, {(-sample_number).bit_length()=}")
print(f"{sample_number.bit_count()=}, {(255).bit_count()=}")

sample_number=176, sample_number.bit_length()=8, (-sample_number).bit_length()=8
sample_number.bit_count()=3, (255).bit_count()=8


`to_bytes(length, byteorder, signed=...)` and `int.from_bytes` convert between an `int`
and its raw bytes, e.g. for binary file formats and network protocols. Both default to
big-endian (3.11+).

In [52]:
port_number = 8080
port_bytes = port_number.to_bytes(2, "big")
print(f"{port_bytes=}, {port_bytes.hex()=}")
print(f"{int.from_bytes(port_bytes, 'big')=}")
print(f"{int.from_bytes(port_bytes, 'little')=}  (byte order matters)")
print(f"{(-1).to_bytes(2, 'big', signed=True)=}")
try:
    (-1).to_bytes(2, "big")
except OverflowError as error:
    print(f"{type(error).__name__}: {error}")

port_bytes=b'\x1f\x90', port_bytes.hex()='1f90'
int.from_bytes(port_bytes, 'big')=8080
int.from_bytes(port_bytes, 'little')=36895  (byte order matters)
(-1).to_bytes(2, 'big', signed=True)=b'\xff\xff'
OverflowError: can't convert negative int to unsigned


`bin`, `oct` and `hex` return prefixed strings. `format()` with `b`, `o`, `x` does the
same without the prefix, and can pad.

In [53]:
print(f"{bin(10)=}, {oct(493)=}, {hex(16_750_899)=}")
print(f"{format(10, '08b')=}, {format(255, 'X')=}")

bin(10)='0b1010', oct(493)='0o755', hex(16_750_899)='0xff9933'
format(10, '08b')='00001010', format(255, 'X')='FF'


`int(text, base)` parses text in bases 2 to 36. Base `0` reads the prefix from the text.
Surrounding whitespace and underscores between digits are allowed.

In [54]:
print(f"{int('ff', 16)=}, {int('1010', 2)=}, {int('z', 36)=}")
print(f"{int('0x1F', 0)=}, {int('0b101', 0)=}")
print(f"{int('  1_000  ')=}")

int('ff', 16)=255, int('1010', 2)=10, int('z', 36)=35
int('0x1F', 0)=31, int('0b101', 0)=5
int('  1_000  ')=1000


`int.is_integer()` (*new in 3.12*) always returns `True`. It exists so code that receives
"a number" can ask the question without checking the type first.

In [55]:
for quantity in (3, 3.0, 3.5):
    print(f"{quantity!r:>4}: {quantity.is_integer()=}")

   3: quantity.is_integer()=True
 3.0: quantity.is_integer()=True
 3.5: quantity.is_integer()=False


**Bitwise operators** work on the binary digits. Their classic use is a set of on/off
flags packed into one integer, such as Unix file permissions. (Phase 2 shows
`enum.Flag`, the type-safe version.)

| Operator | Meaning | Use with flags |
| --- | --- | --- |
| `a \| b` | OR | grant a permission |
| `a & b` | AND | test a permission |
| `a & ~b` | AND NOT | revoke a permission |
| `a ^ b` | XOR | toggle a permission |
| `a << n`, `a >> n` | shift left / right | build masks, multiply / divide by `2**n` |

In [56]:
READ, WRITE, EXECUTE = 0b100, 0b010, 0b001

permissions = READ | WRITE
print(f"{bin(permissions)=}, can write: {bool(permissions & WRITE)}")
permissions &= ~WRITE
print(f"{bin(permissions)=}, can write: {bool(permissions & WRITE)}")
permissions ^= EXECUTE
print(f"{bin(permissions)=}, can execute: {bool(permissions & EXECUTE)}")
print(f"{~5=}  (~x == -x - 1: Python ints behave as infinite two's complement)")
print(f"{1 << 10=}, {1024 >> 3=}")
owner_group_other = (0o7 << 6) | (0o5 << 3) | 0o5
print(f"{oct(owner_group_other)=}")

bin(permissions)='0b110', can write: True
bin(permissions)='0b100', can write: False
bin(permissions)='0b101', can execute: True
~5=-6  (~x == -x - 1: Python ints behave as infinite two's complement)
1 << 10=1024, 1024 >> 3=128
oct(owner_group_other)='0o755'


**The 4300-digit limit (3.11+, also back-ported to security releases).** Converting a
huge `int` to or from **decimal text** takes time quadratic in the number of digits. A
web service that ran `int(request_body)` could be stalled by a megabyte of digits
(CVE-2020-10735). Python therefore refuses decimal conversions above 4300 digits by
default. Binary bases (`hex`, `bin`) are linear and are not limited. Raise the limit
only for trusted data, and put it back afterwards.

In [57]:
huge_number = 10**5000
try:
    str(huge_number)
except ValueError as error:
    print(f"{type(error).__name__}: {str(error)[:70]}...")
print(f"{len(hex(huge_number))=}  (hex is not limited)")

default_limit = sys.get_int_max_str_digits()
sys.set_int_max_str_digits(10_000)
try:
    print(f"{len(str(huge_number))=}")
finally:
    sys.set_int_max_str_digits(default_limit)
print(f"{sys.get_int_max_str_digits()=}")

ValueError: Exceeds the limit (4300 digits) for integer string conversion; use sys...
len(hex(huge_number))=4155  (hex is not limited)
len(str(huge_number))=5001
sys.get_int_max_str_digits()=4300


### 1.3.2 Floating point (`float`)

A Python `float` is an **IEEE 754 double** (64 bits), exactly the same as a JavaScript
`number`. Most decimal fractions such as `0.1` have no exact binary form, so they are
stored as the nearest double, and the errors show up in arithmetic.
`Decimal(some_float)` reveals the value that is really stored. Compare floats with
`math.isclose`, never with `==`.

In [58]:
print(f"{0.1 + 0.2=}")
print(f"{0.1 + 0.2 == 0.3=}")
print(f"{Decimal(0.1)=}")
print(f"{math.isclose(0.1 + 0.2, 0.3)=}")
print(f"{math.isclose(1e-12, 0.0)=}  (relative tolerance fails near 0)")
print(f"{math.isclose(1e-12, 0.0, abs_tol=1e-9)=}")

0.1 + 0.2=0.30000000000000004
0.1 + 0.2 == 0.3=False
Decimal(0.1)=Decimal('0.1000000000000000055511151231257827021181583404541015625')
math.isclose(0.1 + 0.2, 0.3)=True
math.isclose(1e-12, 0.0)=False  (relative tolerance fails near 0)
math.isclose(1e-12, 0.0, abs_tol=1e-9)=True


**`round()` surprises.** Two effects combine:

1. **Banker's rounding:** an exact tie goes to the **even** neighbour, so `round(2.5) == 2`.
   This avoids an upward bias when many values are rounded. JS `Math.round(2.5)` is `3`.
2. **Representation error:** `2.675` is really stored as `2.67499999…`, so it is not a
   tie at all and rounds *down*.

In [59]:
print(f"{round(0.5)=}, {round(1.5)=}, {round(2.5)=}, {round(-2.5)=}")
print(f"{round(2.675, 2)=}")
print(f"{Decimal(2.675)=}")

round(0.5)=0, round(1.5)=2, round(2.5)=2, round(-2.5)=-2
round(2.675, 2)=2.67
Decimal(2.675)=Decimal('2.67499999999999982236431605997495353221893310546875')


**Special values.** `inf`, `-inf` and `nan` ("not a number") come from `float("inf")` /
`math.inf` / `math.nan` or from overflowing operations. `nan` is not equal to anything,
not even itself, so test it with `math.isnan`. Unlike JS, `1 / 0` raises instead of
returning `Infinity`.

In [60]:
positive_infinity = float("inf")
not_a_number = float("nan")
print(f"{positive_infinity > 10**308=}, {-math.inf < -1e308=}")
print(f"{positive_infinity - positive_infinity=}")
print(f"{not_a_number == not_a_number=}, {not_a_number != not_a_number=}")
print(f"{math.isnan(not_a_number)=}, {math.isinf(-math.inf)=}, {math.isfinite(1e308)=}")
print(f"{1e308 * 10=}")
try:
    ratio = 1 / 0
except ZeroDivisionError as error:
    print(f"{type(error).__name__}: {error}")

positive_infinity > 10**308=True, -math.inf < -1e308=True
positive_infinity - positive_infinity=nan
not_a_number == not_a_number=False, not_a_number != not_a_number=True
math.isnan(not_a_number)=True, math.isinf(-math.inf)=True, math.isfinite(1e308)=True
1e308 * 10=inf
ZeroDivisionError: division by zero


**`float` methods.** `as_integer_ratio()` returns the exact fraction the double stores.
For `0.1` that is not `1/10`.

In [61]:
print(f"{(0.75).as_integer_ratio()=}")
print(f"{(0.1).as_integer_ratio()=}")

(0.75).as_integer_ratio()=(3, 4)
(0.1).as_integer_ratio()=(3602879701896397, 36028797018963968)


`is_integer()` tells whether the float has no fractional part.

In [62]:
print(f"{(4.0).is_integer()=}, {(4.5).is_integer()=}, {(1e20).is_integer()=}")

(4.0).is_integer()=True, (4.5).is_integer()=False, (1e20).is_integer()=True


`hex()` and `float.fromhex()` write and read the exact bits in hexadecimal, with no
decimal rounding. Use them to store or send a float losslessly.

In [63]:
exact_text = (0.1).hex()
print(f"{exact_text=}")
print(f"{float.fromhex(exact_text) == 0.1=}")

exact_text='0x1.999999999999ap-4'
float.fromhex(exact_text) == 0.1=True


`sys.float_info` describes the platform's doubles. `epsilon` is the gap between `1.0`
and the next float, and `dig` is how many decimal digits are always kept.

In [64]:
print(f"{sys.float_info.max=}")
print(f"{sys.float_info.min=}")
print(f"{sys.float_info.epsilon=}")
print(f"{sys.float_info.dig=}, {sys.float_info.mant_dig=}")
print(f"{1.0 + sys.float_info.epsilon > 1.0=}, {1.0 + sys.float_info.epsilon / 2 > 1.0=}")

sys.float_info.max=1.7976931348623157e+308
sys.float_info.min=2.2250738585072014e-308
sys.float_info.epsilon=2.220446049250313e-16
sys.float_info.dig=15, sys.float_info.mant_dig=53
1.0 + sys.float_info.epsilon > 1.0=True, 1.0 + sys.float_info.epsilon / 2 > 1.0=False


**Useful `math` functions.** `floor` rounds down, `ceil` rounds up, `trunc` drops the
fraction (towards zero, like `int()`). They differ only for negative numbers, and all
three return `int`.

In [65]:
for value in (3.7, -3.7):
    rounded_values = (math.floor(value), math.ceil(value), math.trunc(value), int(value))
    print(f"{value:>5}: floor, ceil, trunc, int = {rounded_values}")

  3.7: floor, ceil, trunc, int = (3, 4, 3, 3)
 -3.7: floor, ceil, trunc, int = (-4, -3, -3, -3)


`math.sqrt` returns a float and rejects negatives (use `cmath`, 1.3.5). `math.isqrt`
gives the exact integer square root of any `int`.

In [66]:
print(f"{math.sqrt(2)=}, {math.isqrt(10**20 + 1)=}")
try:
    math.sqrt(-1)
except ValueError as error:
    print(f"{type(error).__name__}: {error}")

math.sqrt(2)=1.4142135623730951, math.isqrt(10**20 + 1)=10000000000
ValueError: expected a nonnegative input, got -1.0


`math.fsum` adds floats while tracking every lost low-order bit, so the result is
correctly rounded. A plain `+=` loop (what JS `reduce` does) accumulates error.
*Changed in 3.12:* `sum()` of floats uses compensated (Neumaier) summation, which fixes
the common cases, but `fsum` is still the exact one when values of wildly different
sizes cancel.

In [67]:
ten_dimes = [0.1] * 10
loop_total = 0.0
for dime in ten_dimes:
    loop_total += dime
print(f"{loop_total=}  (naive left-to-right)")
print(f"{sum(ten_dimes)=}, {math.fsum(ten_dimes)=}")

mixed_magnitudes = [1e100, 1.0, -1e100, 1e-100, 1e50, -1.0, -1e50]
print(f"{sum(mixed_magnitudes)=}, {math.fsum(mixed_magnitudes)=}")

loop_total=0.9999999999999999  (naive left-to-right)
sum(ten_dimes)=1.0, math.fsum(ten_dimes)=1.0
sum(mixed_magnitudes)=0.0, math.fsum(mixed_magnitudes)=1e-100


`math.prod` multiplies an iterable, the way `sum` adds one (3.8+).

In [68]:
print(f"{math.prod([2, 3, 7])=}")
print(f"{math.prod([1.05] * 3)=}  (three years of 5% growth)")

math.prod([2, 3, 7])=42
math.prod([1.05] * 3)=1.1576250000000001  (three years of 5% growth)


`math.pi`, `math.e` (and `math.tau`, `math.inf`, `math.nan`) are float constants.

In [69]:
circle_radius_in_metres = 2.5
print(f"{math.pi=}, {math.e=}, {math.tau == 2 * math.pi=}")
print(f"{math.pi * circle_radius_in_metres**2=}")

math.pi=3.141592653589793, math.e=2.718281828459045, math.tau == 2 * math.pi=True
math.pi * circle_radius_in_metres**2=19.634954084936208


### 1.3.3 Exact decimals (`decimal.Decimal`)

`Decimal` stores numbers in base 10, so `0.1` is exactly one tenth. That is what money
needs. **Always construct it from a string (or an `int`), never from a float**: a float
argument copies the float's representation error.

In [70]:
print(f"{Decimal('0.1')=}")
print(f"{Decimal(0.1)=}")
print(f"{Decimal('0.1') + Decimal('0.2') == Decimal('0.3')=}")
print(f"{Decimal(25)=}, {Decimal('1.50')=}  (trailing zeros are kept)")

Decimal('0.1')=Decimal('0.1')
Decimal(0.1)=Decimal('0.1000000000000000055511151231257827021181583404541015625')
Decimal('0.1') + Decimal('0.2') == Decimal('0.3')=True
Decimal(25)=Decimal('25'), Decimal('1.50')=Decimal('1.50')  (trailing zeros are kept)


**Arithmetic.** `Decimal` supports the usual operators and mixes with `int`, but not
with `float` (that would bring the inexactness back). Results are rounded to the
**context precision**: 28 significant digits by default. `localcontext()` changes the
precision (or the rounding) only inside a `with` block.

In [71]:
unit_price = Decimal("19.99")
print(f"{unit_price * 3=}, {unit_price - 5=}, {unit_price / 4=}")
print(f"{Decimal(1) / Decimal(7)=}")
with localcontext() as local_context:
    local_context.prec = 6
    print(f"inside localcontext: {Decimal(1) / Decimal(7)=}")
print(f"after the block: {getcontext().prec=}")
try:
    unit_price + 0.5
except TypeError as error:
    print(f"{type(error).__name__}: {error}")

unit_price * 3=Decimal('59.97'), unit_price - 5=Decimal('14.99'), unit_price / 4=Decimal('4.9975')


Decimal(1) / Decimal(7)=Decimal('0.1428571428571428571428571429')
inside localcontext: Decimal(1) / Decimal(7)=Decimal('0.142857')
after the block: getcontext().prec=28
TypeError: unsupported operand type(s) for +: 'decimal.Decimal' and 'float'


**`quantize`** rounds to a fixed exponent, e.g. `Decimal("0.01")` for paise. The
rounding mode is explicit:

| Mode | Ties go… | Typical use |
| --- | --- | --- |
| `ROUND_HALF_UP` | away from zero (`2.675 → 2.68`) | invoices and tax, as taught in school |
| `ROUND_HALF_EVEN` | to the even digit (`2.665 → 2.66`) | statistics, the `Decimal` default |

In [72]:
print(f"{'value':>7} {'HALF_UP':>8} {'HALF_EVEN':>10}")
for amount_text in ["2.675", "2.665", "0.125", "-1.005"]:
    amount = Decimal(amount_text)
    half_up = amount.quantize(PAISE, rounding=ROUND_HALF_UP)
    half_even = amount.quantize(PAISE, rounding=ROUND_HALF_EVEN)
    print(f"{amount_text:>7} {half_up!s:>8} {half_even!s:>10}")

  value  HALF_UP  HALF_EVEN
  2.675     2.68       2.68
  2.665     2.67       2.66
  0.125     0.13       0.12
 -1.005    -1.01      -1.00


**Money: an 18% GST demo.** One thousand orders of ₹99.99 each, with GST computed and
rounded per order. Float accumulates representation error across the additions;
`Decimal` is exact to the paisa.

In [73]:
order_count = 1_000

float_grand_total = 0.0
for _ in range(order_count):
    float_grand_total += 99.99 + round(99.99 * 0.18, 2)

decimal_grand_total = Decimal("0.00")
for _ in range(order_count):
    gst = (Decimal("99.99") * Decimal("0.18")).quantize(PAISE, rounding=ROUND_HALF_UP)
    decimal_grand_total += Decimal("99.99") + gst

print(f"{float_grand_total=}")
print(f"{decimal_grand_total=}")

float_grand_total=117990.00000000196
decimal_grand_total=Decimal('117990.00')


**Performance trade-off.** `float` arithmetic runs in hardware; `Decimal` is software
(C-accelerated `_decimal`, but still slower). The usual pattern is `Decimal` (or
integer paise) for money and `float` for measurements and science. Timings vary by
machine; the ratio is what matters.

In [74]:
float_operands = {"a": 99.99, "b": 0.18, "c": 1.5}
decimal_operands = {"a": Decimal("99.99"), "b": Decimal("0.18"), "c": Decimal("1.5")}
float_seconds = timeit.timeit("a * b + c", globals=float_operands, number=200_000)
decimal_seconds = timeit.timeit("a * b + c", globals=decimal_operands, number=200_000)
slowdown = decimal_seconds / float_seconds
print(f"float:   {float_seconds * 1000:.1f} ms")
print(f"Decimal: {decimal_seconds * 1000:.1f} ms  (about {slowdown:.0f}x slower)")

float:   3.8 ms
Decimal: 17.1 ms  (about 5x slower)


### 1.3.4 Rational numbers (`fractions.Fraction`)

A `Fraction` is an exact numerator/denominator pair, always kept in lowest terms. One
third really is one third, so `Fraction(1, 3) * 3 == 1`.

In [75]:
one_third = Fraction(1, 3)
print(f"{one_third * 3 == 1=}, {one_third * 3=}")
print(f"{Fraction(6, 8)=}  (reduced automatically)")
print(f"{one_third + Fraction(1, 6)=}")
print(f"{1 / 3 * 3 == 1=}  (float happens to round back here, but not in general)")

one_third * 3 == 1=True, one_third * 3=Fraction(1, 1)
Fraction(6, 8)=Fraction(3, 4)  (reduced automatically)
one_third + Fraction(1, 6)=Fraction(1, 2)
1 / 3 * 3 == 1=True  (float happens to round back here, but not in general)


Build a `Fraction` from a **string** for the exact value you typed. Building it from a
float shows the float's exact binary value. `limit_denominator(max_denominator)` finds
the closest fraction with a small denominator, which recovers the intended value.

In [76]:
print(f"{Fraction('3/7')=}, {Fraction('0.125')=}, {Fraction('1.5e-2')=}")
print(f"{Fraction(0.1)=}")
print(f"{Fraction(0.1).limit_denominator(1000)=}")
print(f"{Fraction(math.pi).limit_denominator(1000)=}")

Fraction('3/7')=Fraction(3, 7), Fraction('0.125')=Fraction(1, 8), Fraction('1.5e-2')=Fraction(3, 200)
Fraction(0.1)=Fraction(3602879701896397, 36028797018963968)
Fraction(0.1).limit_denominator(1000)=Fraction(1, 10)
Fraction(math.pi).limit_denominator(1000)=Fraction(355, 113)


### 1.3.5 Complex numbers (`complex`)

Python has complex numbers built in: write the imaginary part with a `j` suffix. They
are used in signal processing and electrical engineering; you will rarely need them for
web work.

In [77]:
impedance = 3 + 4j
print(f"{impedance=}, {type(impedance).__name__=}")
print(f"{impedance.real=}, {impedance.imag=}")
print(f"{abs(impedance)=}  (magnitude)")
print(f"{impedance.conjugate()=}, {impedance * impedance.conjugate()=}")
print(f"{complex(3, 4) == impedance=}")

impedance=(3+4j), type(impedance).__name__='complex'
impedance.real=3.0, impedance.imag=4.0
abs(impedance)=5.0  (magnitude)
impedance.conjugate()=(3-4j), impedance * impedance.conjugate()=(25+0j)
complex(3, 4) == impedance=True


`cmath` is the complex-number version of `math`: its functions accept negative and
complex inputs.

In [78]:
print(f"{cmath.sqrt(-1)=}")
print(f"{cmath.phase(1j)=}  (pi / 2)")
print(f"{cmath.polar(impedance)=}")
print(f"{cmath.rect(5, 0.9272952180016122)=}")

cmath.sqrt(-1)=1j
cmath.phase(1j)=1.5707963267948966  (pi / 2)
cmath.polar(impedance)=(5.0, 0.9272952180016122)
cmath.rect(5, 0.9272952180016122)=(3.0000000000000004+3.9999999999999996j)


### 1.3.6 Booleans (`bool`)

`bool` is a **subclass of `int`**: `True` is `1` and `False` is `0` in arithmetic. A
useful consequence: `sum(flags)` counts the `True` values.

In [79]:
print(f"{issubclass(bool, int)=}")
print(f"{True + True=}, {True * 10=}, {True == 1=}")
order_is_paid = [True, False, True, True]
print(f"{sum(order_is_paid)=} paid orders")

issubclass(bool, int)=True
True + True=2, True * 10=10, True == 1=True
sum(order_is_paid)=3 paid orders


**The validation trap.** Because `True` *is* an `int`, `isinstance(True, int)` is
`True`, and a check meant to accept quantities also accepts booleans. Reject `bool`
explicitly.

In [80]:
def is_valid_quantity_naive(quantity: object) -> bool:
    """Accept positive ints (buggy: also accepts True)."""
    return isinstance(quantity, int) and quantity > 0


def is_valid_quantity(quantity: object) -> bool:
    """Accept positive ints, rejecting bool."""
    return isinstance(quantity, int) and not isinstance(quantity, bool) and quantity > 0


for candidate in (3, True, 0, 2.0):
    naive_verdict = is_valid_quantity_naive(candidate)
    print(f"{candidate!r:>5}: naive={naive_verdict}, fixed={is_valid_quantity(candidate)}")

    3: naive=True, fixed=True
 True: naive=True, fixed=False
    0: naive=False, fixed=False
  2.0: naive=False, fixed=False


`bool(x)` converts any value using the truthiness rules (1.3.8). Watch out for strings:
any non-empty string is `True`, including `"False"` and `"0"`.

In [81]:
print(f"{bool(0)=}, {bool(42)=}, {bool('')=}, {bool('False')=}, {bool('0')=}")
form_value = "false"
print(f"{form_value.strip().lower() in {'true', '1', 'yes'}=}  (parse text explicitly)")

bool(0)=False, bool(42)=True, bool('')=False, bool('False')=True, bool('0')=True
form_value.strip().lower() in {'true', '1', 'yes'}=False  (parse text explicitly)


### 1.3.7 The null value (`None`)

`None` is the only instance of `NoneType`, so every `None` is the same object; check it
with `is None`. A function without a `return` (or with a bare `return`) returns `None`.

In [82]:
def log_message(message: str) -> None:
    """Print a message; no return statement."""
    print(f"log: {message}")


result = log_message("order created")
print(f"{result is None=}")
print(f"{type(None).__name__=}, {type(None)() is None=}")

log: order created


result is None=True
type(None).__name__='NoneType', type(None)() is None=True


> **JS bridge:** Python has **no `undefined`**. An unassigned name raises `NameError`,
> a missing attribute raises `AttributeError`, and a missing dict key raises `KeyError`.
> `dict.get(key)` returns `None` (or a default) when you want JS-like behaviour.

In [83]:
customer = {"name": "Asha", "city": "Pune"}
try:
    customer["phone"]
except KeyError as error:
    print(f"{type(error).__name__}: {error}")
print(f"{customer.get('phone')=}, {customer.get('phone', 'not given')=}")

KeyError: 'phone'
customer.get('phone')=None, customer.get('phone', 'not given')='not given'


### 1.3.8 Truthiness

Every object is true or false in an `if`, `while`, `and`, `or` or `not`. These values are
**falsy**; everything else is truthy:

| Falsy value | Type |
| --- | --- |
| `None` | `NoneType` |
| `False` | `bool` |
| `0`, `0.0`, `0j` | `int`, `float`, `complex` |
| `Decimal(0)`, `Fraction(0)` | `Decimal`, `Fraction` |
| `""`, `b""` | `str`, `bytes` |
| `[]`, `()`, `{}` | `list`, `tuple`, `dict` |
| `set()`, `range(0)` | `set`, `range` |
| an object whose `__bool__` returns `False`, or whose `__len__` returns `0` | user classes |

The cell evaluates every row.

In [84]:
falsy_candidates = [None, False, 0, 0.0, 0j, Decimal(0), Fraction(0)]
falsy_candidates += ["", b"", [], (), {}, set(), range(0)]
for candidate in falsy_candidates:
    print(f"{candidate!r:<14} -> {bool(candidate)}")

None           -> False
False          -> False
0              -> False
0.0            -> False
0j             -> False
Decimal('0')   -> False
Fraction(0, 1) -> False
''             -> False
b''            -> False
[]             -> False
()             -> False
{}             -> False
set()          -> False
range(0, 0)    -> False


User classes decide with `__bool__`, or, if it is missing, with `__len__` (zero length
is false). With neither, every instance is truthy.

In [85]:
class Wallet:
    """A wallet that is truthy only when it holds money."""

    def __init__(self, balance_in_paise: int) -> None:
        self.balance_in_paise = balance_in_paise

    def __bool__(self) -> bool:
        return self.balance_in_paise > 0


class ShoppingBag:
    """A bag whose truthiness comes from its length."""

    def __init__(self, items: list[str]) -> None:
        self.items = items

    def __len__(self) -> int:
        return len(self.items)


class PlainObject:
    """No __bool__ and no __len__."""


print(f"{bool(Wallet(0))=}, {bool(Wallet(500))=}")
print(f"{bool(ShoppingBag([]))=}, {bool(ShoppingBag(['soap']))=}")
print(f"{bool(PlainObject())=}")

bool(Wallet(0))=False, bool(Wallet(500))=True
bool(ShoppingBag([]))=False, bool(ShoppingBag(['soap']))=True
bool(PlainObject())=True


**Differences from JavaScript**

| Value | JavaScript | Python |
| --- | --- | --- |
| `[]`, `{}` | truthy | **falsy** |
| `"0"`, `" "` | truthy | truthy |
| `NaN` | falsy | **truthy** (`nan` is a non-zero float) |
| `0n` / `0` | falsy | falsy |
| `document.all` | falsy (!) | — |

**When to prefer `is not None`.** `if value:` treats `0`, `""` and `[]` the same as a
missing value. When `0` is a legitimate input, test for `None` explicitly.

In [86]:
def price_after_discount_buggy(price: int, discount: int | None = None) -> int:
    """Apply a discount; treats a discount of 0 as 'not given' (bug)."""
    if not discount:
        discount = 10  # default discount
    return price - discount


def price_after_discount(price: int, discount: int | None = None) -> int:
    """Apply a discount; only None means 'not given'."""
    if discount is None:
        discount = 10
    return price - discount


print(f"{bool(float('nan'))=}")
print(f"{price_after_discount_buggy(100, 0)=}  (wanted 100)")
print(f"{price_after_discount(100, 0)=}")

bool(float('nan'))=True
price_after_discount_buggy(100, 0)=90  (wanted 100)
price_after_discount(100, 0)=100


### 1.3.9 Type conversion

Conversions are always **explicit**: call the type. They raise `ValueError` when the
text is not a valid literal for that type. Note that `int("3.5")` fails: convert to
`float` first if decimals are allowed. `int(3.99)` truncates.

In [87]:
print(f"{int('42')=}, {int(' -7 ')=}, {int(3.99)=}, {int(True)=}")
print(f"{float('3.5')=}, {float('1e3')=}, {float('nan')=}, {float(7)=}")
print(f"{str(42)=}, {str(3.5)=}, {str(None)=}")
print(f"{bool(1)=}, {bool('')=}")
print(f"{complex('3+4j')=}, {complex(2)=}")
print(f"{int(float('3.5'))=}")

int('42')=42, int(' -7 ')=-7, int(3.99)=3, int(True)=1
float('3.5')=3.5, float('1e3')=1000.0, float('nan')=nan, float(7)=7.0
str(42)='42', str(3.5)='3.5', str(None)='None'
bool(1)=True, bool('')=False
complex('3+4j')=(3+4j), complex(2)=(2+0j)
int(float('3.5'))=3


Each failing conversion raises `ValueError` with a message that quotes the bad text.

In [88]:
for converter, bad_text in [(int, "3.5"), (int, "12abc"), (float, "abc"), (complex, "3 + 4j")]:
    try:
        converter(bad_text)
    except ValueError as error:
        print(f"{converter.__name__}({bad_text!r}) -> {type(error).__name__}: {error}")

int('3.5') -> ValueError: invalid literal for int() with base 10: '3.5'
int('12abc') -> ValueError: invalid literal for int() with base 10: '12abc'
float('abc') -> ValueError: could not convert string to float: 'abc'
complex('3 + 4j') -> ValueError: complex() arg is a malformed string


**No implicit coercion.** JavaScript converts silently (`1 + "1" === "11"`,
`1 == "1"` is `true`). Python raises `TypeError` for mixed `+` and returns `False` for
`==` between unrelated types. Numeric types do mix with each other (`1 + 2.5`, `1 == 1.0`).

In [89]:
try:
    mixed_total = 1 + "1"
except TypeError as error:
    print(f"{type(error).__name__}: {error}")
print(f"{1 == '1'=}")
print(f"{1 + int('1')=}, {str(1) + '1'=}")
print(f"{1 + 2.5=}, {1 == 1.0 == Decimal(1) == Fraction(1)=}")

TypeError: unsupported operand type(s) for +: 'int' and 'str'
1 == '1'=False
1 + int('1')=2, str(1) + '1'='11'
1 + 2.5=3.5, 1 == 1.0 == Decimal(1) == Fraction(1)=True


#### Under the hood: object sizes and the int digit formula

Every number is a full object with a header, so even `True` costs bytes. For `int`,
the size is a fixed header plus 4 bytes per 30-bit digit; the cell checks the formula
against `sys.getsizeof` for growing values. `show_size` (shared setup) prints a few
other numeric types for comparison.

In [90]:
int_header_bytes = sys.getsizeof(1) - sys.int_info.sizeof_digit
for exponent in (10, 100, 1_000):
    number = 2**exponent
    digit_count = -(-number.bit_length() // sys.int_info.bits_per_digit)  # ceiling division
    predicted_bytes = int_header_bytes + digit_count * sys.int_info.sizeof_digit
    actual_bytes = sys.getsizeof(number)
    print(f"2**{exponent:<5} digits={digit_count:<3} {predicted_bytes=:<4} {actual_bytes=}")

show_size("float", 1.5)
show_size("complex", 1 + 2j)
show_size("Decimal", Decimal("1.5"))
show_size("Fraction", Fraction(1, 2))
show_size("True", True)

2**10    digits=1   predicted_bytes=28   actual_bytes=28
2**100   digits=4   predicted_bytes=40   actual_bytes=40
2**1000  digits=34  predicted_bytes=160  actual_bytes=160
float: 24 bytes, type=float, id=139869173723920
complex: 32 bytes, type=complex, id=139868745265808
Decimal: 120 bytes, type=Decimal, id=139868725461952
Fraction: 48 bytes, type=Fraction, id=139868745366736
True: 28 bytes, type=bool, id=22110296


### Lab 1.3

Three exercises: an exact bill splitter, an invoice calculator (with a float version to
show drift), and predict-the-output questions. For each coding exercise, replace the
`raise NotImplementedError` in the stub, then run the check cell. Until then the checks
print FAIL with `NotImplementedError`; that is expected.

#### Exercise 1 — `split_bill`

Write `split_bill(total_amount: str, people_count: int) -> list[Decimal]`:

- Split the total (a rupee amount as text, e.g. `"100"` or `"1000.01"`) into
  `people_count` parts **exact to the paisa**: every part has two decimal places and
  the parts sum to exactly the total.
- Parts differ by at most one paisa; the leftover paise go to the **first** people.
  `split_bill("100", 3) == [Decimal("33.34"), Decimal("33.33"), Decimal("33.33")]`.
- Raise `ValueError` when `people_count` is less than 1.

*Hint:* work in whole paise with `divmod`.

In [91]:
def split_bill(total_amount: str, people_count: int) -> list[Decimal]:
    """Split a rupee amount into people_count parts that sum exactly to the total."""
    raise NotImplementedError("Lab 1.3: implement split_bill()")

The check cell defines `raised_error_name(thunk)`, which returns the name of the
exception a call raises (or `"no error"`), so error cases fit the
`(label, thunk, expected)` format of `run_checks`.

In [92]:
def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


run_checks(
    [
        (
            "100 between 3",
            lambda: split_bill("100", 3),
            [Decimal("33.34"), Decimal("33.33"), Decimal("33.33")],
        ),
        ("parts sum to the total", lambda: sum(split_bill("1000.01", 7)), Decimal("1000.01")),
        ("one part per person", lambda: len(split_bill("10", 4)), 4),
        (
            "parts differ by at most one paisa",
            lambda: max(split_bill("0.05", 3)) - min(split_bill("0.05", 3)) <= PAISE,
            True,
        ),
        (
            "every part has two decimal places",
            lambda: {part.as_tuple().exponent for part in split_bill("90", 4)},
            {-2},
        ),
        ("returns Decimal", lambda: type(split_bill("1", 1)[0]).__name__, "Decimal"),
        (
            "zero people raises",
            lambda: raised_error_name(lambda: split_bill("10", 0)),
            "ValueError",
        ),
    ]
)

FAIL  100 between 3: raised NotImplementedError: Lab 1.3: implement split_bill()
FAIL  parts sum to the total: raised NotImplementedError: Lab 1.3: implement split_bill()
FAIL  one part per person: raised NotImplementedError: Lab 1.3: implement split_bill()
FAIL  parts differ by at most one paisa: raised NotImplementedError: Lab 1.3: implement split_bill()
FAIL  every part has two decimal places: raised NotImplementedError: Lab 1.3: implement split_bill()
FAIL  returns Decimal: raised NotImplementedError: Lab 1.3: implement split_bill()
FAIL  zero people raises: expected 'ValueError', got 'NotImplementedError'
0/7 checks passed


#### Exercise 2 — invoice calculator

An invoice is a list of `(unit_price, quantity)` line items. For **each line**:
net amount = unit price × quantity, GST = net amount × rate, **rounded to paise per
line**. The invoice total is the sum of every line's net amount plus its GST.

1. `invoice_total_decimal(line_items: list[tuple[str, int]], gst_rate_percent: str = "18") -> Decimal`:
   prices and rate are strings, GST is rounded with `quantize(PAISE, rounding=ROUND_HALF_UP)`,
   and the total is returned with two decimal places.
2. `invoice_total_float(line_items: list[tuple[float, int]], gst_rate_percent: float = 18.0) -> float`:
   the same calculation with floats and `round(gst, 2)`. Do **not** round the final
   total, so the drift stays visible.

In [93]:
def invoice_total_decimal(
    line_items: list[tuple[str, int]], gst_rate_percent: str = "18"
) -> Decimal:
    """Return net + per-line GST (ROUND_HALF_UP to paise) for all line items."""
    raise NotImplementedError("Lab 1.3: implement invoice_total_decimal()")


def invoice_total_float(
    line_items: list[tuple[float, int]], gst_rate_percent: float = 18.0
) -> float:
    """Return the same total computed with float and round()."""
    raise NotImplementedError("Lab 1.3: implement invoice_total_float()")

The checks include a line of ₹2.75: its GST is exactly ₹0.495, which `ROUND_HALF_UP`
makes ₹0.50. The float `2.75 * 18 / 100` is stored just *below* 0.495, so `round()`
gives ₹0.49.

In [94]:
sample_line_items = [
    ("19.99", 3),
    ("0.10", 7),
    ("2.75", 1),
    ("12.25", 2),
    ("0.25", 1),
    ("5.25", 1),
]
float_line_items = [(float(price), quantity) for price, quantity in sample_line_items]

run_checks(
    [
        ("empty invoice", lambda: invoice_total_decimal([]), Decimal("0.00")),
        ("one line, 18% GST", lambda: invoice_total_decimal([("100", 2)]), Decimal("236.00")),
        ("GST rounds half up", lambda: invoice_total_decimal([("1.25", 1)]), Decimal("1.48")),
        ("custom rate 5%", lambda: invoice_total_decimal([("10.10", 1)], "5"), Decimal("10.61")),
        ("sample invoice", lambda: invoice_total_decimal(sample_line_items), Decimal("110.25")),
        ("float: one line, 18% GST", lambda: invoice_total_float([(100.0, 2)]), 236.0),
        ("float: round() loses a paisa on 2.75", lambda: invoice_total_float([(2.75, 1)]), 3.24),
        (
            "float: sample invoice drifts from the exact total",
            lambda: (
                invoice_total_float(float_line_items)
                != float(invoice_total_decimal(sample_line_items))
            ),
            True,
        ),
    ]
)

FAIL  empty invoice: raised NotImplementedError: Lab 1.3: implement invoice_total_decimal()
FAIL  one line, 18% GST: raised NotImplementedError: Lab 1.3: implement invoice_total_decimal()
FAIL  GST rounds half up: raised NotImplementedError: Lab 1.3: implement invoice_total_decimal()
FAIL  custom rate 5%: raised NotImplementedError: Lab 1.3: implement invoice_total_decimal()
FAIL  sample invoice: raised NotImplementedError: Lab 1.3: implement invoice_total_decimal()
FAIL  float: one line, 18% GST: raised NotImplementedError: Lab 1.3: implement invoice_total_float()
FAIL  float: round() loses a paisa on 2.75: raised NotImplementedError: Lab 1.3: implement invoice_total_float()
FAIL  float: sample invoice drifts from the exact total: raised NotImplementedError: Lab 1.3: implement invoice_total_float()
0/8 checks passed


Once both functions work, this cell shows the drift side by side. Until then it prints
a reminder.

In [95]:
try:
    exact_invoice_total = invoice_total_decimal(sample_line_items)
    float_invoice_total = invoice_total_float(float_line_items)
except NotImplementedError as error:
    print(f"Not yet: {error}")
else:
    print(f"{exact_invoice_total=}")
    print(f"{float_invoice_total=}")
    print(f"difference: {Decimal(float_invoice_total) - exact_invoice_total:.17f}")

Not yet: Lab 1.3: implement invoice_total_decimal()


#### Exercise 3 — predict the output

**Predict the output.**

```python
print(-7 // 2, 7 // -2, -7.0 // 2)
```

<details><summary>Answer</summary>

`-4 -4 -4.0`. `//` floors towards negative infinity: −3.5 becomes −4 in both sign
combinations. With a float operand the result is a float.

</details>

Run the next cell to check your prediction.

In [96]:
print(-7 // 2, 7 // -2, -7.0 // 2)

-4 -4 -4.0


**Predict the output.**

```python
print(-7 % 2, 7 % -2, -7 % -2)
```

<details><summary>Answer</summary>

`1 -1 -1`. The result of `%` takes the sign of the **divisor**, so that
`(a // b) * b + a % b == a`. For `-7 % 2`: `(-4) * 2 + 1 == -7`.

</details>

Run the next cell to check your prediction.

In [97]:
print(-7 % 2, 7 % -2, -7 % -2)

1 -1 -1


**Predict the output.**

```python
print(round(0.5), round(1.5), round(2.5), round(2.675, 2))
```

<details><summary>Answer</summary>

`0 2 2 2.67`. Exact ties go to the even neighbour (banker's rounding), and `2.675` is
stored as `2.67499999…`, which is below the tie.

</details>

Run the next cell to check your prediction.

In [98]:
print(round(0.5), round(1.5), round(2.5), round(2.675, 2))

0 2 2 2.67


**Predict the output.**

```python
print(bool([0]), bool("0"), bool(0.0), bool(None), bool(" "), bool({}))
```

<details><summary>Answer</summary>

`True True False False True False`. `[0]` is a non-empty list, and `"0"` and `" "` are
non-empty strings, so all three are truthy. `0.0`, `None` and the empty dict `{}` are
falsy.

</details>

Run the next cell to check your prediction.

In [99]:
print(bool([0]), bool("0"), bool(0.0), bool(None), bool(" "), bool({}))

True True False False True False


### Recap — Unit 1.3

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| `int` | Arbitrary precision, 30-bit digits, never overflows | `BigInt` |
| `/`, `//`, `%` | `/` is always float; `//` floors; `%` takes the divisor's sign | `/`, `Math.floor`, remainder `%` |
| 4300-digit limit | Decimal-text conversion of huge ints is capped against DoS | — |
| `float` | IEEE 754 double; compare with `math.isclose`; `1/0` raises | `number` |
| `round()` | Round half to even, plus representation error | `Math.round` rounds half up |
| `Decimal` | Exact base-10; build from strings; `quantize` with explicit rounding for money | decimal.js, big.js |
| `Fraction` | Exact rationals; `limit_denominator` recovers simple ratios | — |
| `complex` | `3 + 4j`, `.real`, `.imag`, `abs`, `cmath` | — |
| `bool` | Subclass of `int`; reject it explicitly when validating ints | `boolean` |
| `None` | The only null; check with `is None`; no `undefined` | `null` / `undefined` |
| Truthiness | Empty containers and zeros are falsy; use `is not None` when 0 is valid | `[]` and `{}` are truthy |
| Conversion | Always explicit; bad text raises `ValueError`; mixed `+` raises `TypeError` | Implicit coercion |

**Further reading**

- [Numeric types — `int`, `float`, `complex`](https://docs.python.org/3/library/stdtypes.html#numeric-types-int-float-complex),
  [integer string conversion length limitation](https://docs.python.org/3/library/stdtypes.html#int-max-str-digits)
- [Floating-point arithmetic: issues and limitations](https://docs.python.org/3/tutorial/floatingpoint.html),
  [`math`](https://docs.python.org/3/library/math.html),
  [`cmath`](https://docs.python.org/3/library/cmath.html)
- [`decimal`](https://docs.python.org/3/library/decimal.html),
  [`fractions`](https://docs.python.org/3/library/fractions.html)
- [Truth value testing](https://docs.python.org/3/library/stdtypes.html#truth-value-testing),
  [`None`](https://docs.python.org/3/library/constants.html#None)
- [PEP 515 — Underscores in numeric literals](https://peps.python.org/pep-0515/),
  [PEP 285 — Adding a bool type](https://peps.python.org/pep-0285/),
  [PEP 327 — Decimal data type](https://peps.python.org/pep-0327/),
  [PEP 237 — Unifying long integers and integers](https://peps.python.org/pep-0237/)